In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import pickle

df = pd.read_csv('../data/processed/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("Rows:", df.shape[0])
print("Ready for recommendations!")

Rows: 805549
Ready for recommendations!


In [2]:
# Count how many times each customer bought each product
customer_product = df.groupby(
    ['Customer ID', 'Description'])['Quantity'].sum().unstack(fill_value=0)

print("Matrix shape:", customer_product.shape)
print("Rows = Customers, Columns = Products")

Matrix shape: (5878, 5283)
Rows = Customers, Columns = Products


In [3]:
# Normalize so heavy buyers don't dominate
matrix_normalized = normalize(customer_product.values)

print("Normalized matrix shape:", matrix_normalized.shape)

Normalized matrix shape: (5878, 5283)


In [4]:
# Cosine similarity between all customers
customer_similarity = cosine_similarity(matrix_normalized)
customer_similarity_df = pd.DataFrame(
    customer_similarity,
    index=customer_product.index,
    columns=customer_product.index
)

print("Similarity matrix shape:", customer_similarity_df.shape)
print("Sample:")
print(customer_similarity_df.iloc[:3, :3])

Similarity matrix shape: (5878, 5878)
Sample:
Customer ID  12346.0   12347.0   12348.0
Customer ID                             
12346.0          1.0  0.000000  0.000000
12347.0          0.0  1.000000  0.150163
12348.0          0.0  0.150163  1.000000


In [5]:
def recommend_products(customer_id, n_recommendations=5):
    # Check if customer exists
    if customer_id not in customer_similarity_df.index:
        return f"Customer {customer_id} not found in dataset."

    # Get top 10 similar customers (exclude self)
    similar_customers = customer_similarity_df[customer_id]\
        .sort_values(ascending=False)\
        .drop(customer_id)\
        .head(10)\
        .index.tolist()

    # Products already bought by this customer
    already_bought = set(customer_product.loc[customer_id][
        customer_product.loc[customer_id] > 0].index)

    # Collect products bought by similar customers
    recommended = {}
    for similar_customer in similar_customers:
        their_products = customer_product.loc[similar_customer]
        for product, qty in their_products.items():
            if qty > 0 and product not in already_bought:
                recommended[product] = recommended.get(product, 0) + qty

    # Sort by most recommended
    recommended_sorted = sorted(recommended.items(),
        key=lambda x: x[1], reverse=True)

    return [product for product, _ in recommended_sorted[:n_recommendations]]

print("✅ Recommendation function ready!")

✅ Recommendation function ready!


In [6]:
# Pick 5 random customers to test
test_customers = customer_product.index[:5].tolist()

print("===== RECOMMENDATION TEST =====\n")
for customer_id in test_customers:
    recommendations = recommend_products(customer_id)
    print(f"Customer {customer_id}:")
    for i, product in enumerate(recommendations, 1):
        print(f"  {i}. {product}")
    print()

===== RECOMMENDATION TEST =====

Customer 12346.0:
  1. PARTY BUNTING
  2. SPOTTY BUNTING
  3. WRAP ALPHABET DESIGN
  4. CUPCAKE LACE PAPER SET 6
  5. PAPER BUNTING WHITE LACE

Customer 12347.0:
  1. PACK OF 72 RETROSPOT CAKE CASES
  2. DOUGHNUT LIP GLOSS 
  3. 60 CAKE CASES DOLLY GIRL DESIGN
  4. LUNCH BAG RED RETROSPOT
  5. PAPER CHAIN KIT RETRO SPOT

Customer 12348.0:
  1. PACK OF 12 PINK PAISLEY TISSUES 
  2. PACK OF 60 DINOSAUR CAKE CASES
  3. PACK OF 72 RETROSPOT CAKE CASES
  4. PACK OF 72 RETRO SPOT CAKE CASES
  5. PACK OF 12 PINK SPOT TISSUES 

Customer 12349.0:
  1. HEART OF WICKER SMALL
  2. ASSORTED CAKES FRIDGE MAGNETS
  3. MINI CAKE STAND  HANGING STRAWBERY
  4. HEART OF WICKER LARGE
  5. HEART FILIGREE DOVE  SMALL

Customer 12350.0:
  1. WORLD WAR 2 GLIDERS ASSTD DESIGNS
  2. BALLOON WATER BOMB PACK OF 35
  3. TRADITIONAL WOODEN SKIPPING ROPE
  4. GROW A FLYTRAP OR SUNFLOWER IN TIN
  5. PAPER CHAIN KIT 50'S CHRISTMAS 



In [7]:
# Item based: customers who bought X also bought Y
product_similarity = cosine_similarity(matrix_normalized.T)
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product.columns,
    columns=customer_product.columns
)

def similar_products(product_name, n=5):
    if product_name not in product_similarity_df.index:
        return f"Product '{product_name}' not found."

    similar = product_similarity_df[product_name]\
        .sort_values(ascending=False)\
        .drop(product_name)\
        .head(n)

    return similar.index.tolist()

# Test it
test_product = 'REGENCY CAKESTAND 3 TIER'
print(f"Customers who bought '{test_product}' also bought:\n")
for i, p in enumerate(similar_products(test_product), 1):
    print(f"  {i}. {p}")

Customers who bought 'REGENCY CAKESTAND 3 TIER' also bought:

  1. ROSES REGENCY TEACUP AND SAUCER 
  2. GREEN REGENCY TEACUP AND SAUCER
  3. PINK REGENCY TEACUP AND SAUCER
  4. SET OF 3 REGENCY CAKE TINS
  5. REGENCY TEAPOT ROSES 


In [8]:
with open('../models/recommendation_model.pkl', 'wb') as f:
    pickle.dump(customer_similarity_df, f)

with open('../models/item_similarity_model.pkl', 'wb') as f:
    pickle.dump(product_similarity_df, f)

# Save customer product matrix for Streamlit app
customer_product.to_csv('../data/processed/customer_product_matrix.csv')

print("✅ Saved to models/recommendation_model.pkl")
print("✅ Saved to models/item_similarity_model.pkl")
print("✅ Saved to data/processed/customer_product_matrix.csv")

✅ Saved to models/recommendation_model.pkl
✅ Saved to models/item_similarity_model.pkl
✅ Saved to data/processed/customer_product_matrix.csv
